## Modele specializate pe recomandari

1. DistilBERT – bazat pe Q&A

In [1]:
from transformers import pipeline

qa = pipeline("question-answering",model="distilbert-base-cased-distilled-squad")

context1_1 = "Best chocolate treat is a fudgy brownie"
question1_1 = "What cake would you recommend if I'm craving chocolate?"

context1_2 = "Daca ai pofte dulci si vrei ceva cu ciocolata, iti recomand Tort Trio Chocolate."
question1_2 = "Recomanda-mi ceva cu ciocolata"

print(qa(question=question1_1, context=context1_1))
print(qa(question=question1_2,context=context1_2))

ModuleNotFoundError: No module named 'transformers'

2. RoBERTa QA

In [12]:
qa = pipeline("question-answering", model="deepset/roberta-base-squad2")

context2_1 = "If you are craving something fruity, we recommend fruit tarts and for chocolate lovers, we recommend our Mozart's Cake."
question2_1_1 = "What should I order with strawberry?"
question2_1_2 = "And if I want chocolaty?"

context2_2 = "Daca clientul isi doreste un dulce cu fructe recomandam tarta cu fructe, iar daca vrea ceva cu ciocolata recomandam brownie cu ciocolata amara."
question2_2_1 = "Recomanda-mi ceva cu piersici"
question2_2_2 = "Recomanda-mi ceva cu ciocolata"

print(qa(question=question2_1_1, context=context2_1))
print(qa(question=question2_1_2, context=context2_1))

print(qa(question=question2_2_1, context=context2_2))
print(qa(question=question2_2_2, context=context2_2))

Device set to use cpu


{'score': 0.020914511755108833, 'start': 50, 'end': 61, 'answer': 'fruit tarts'}
{'score': 0.15872107446193695, 'start': 105, 'end': 118, 'answer': "Mozart's Cake"}
{'score': 1.8295955669600517e-05, 'start': 95, 'end': 104, 'answer': 'ciocolata'}
{'score': 0.000900601502507925, 'start': 105, 'end': 142, 'answer': 'recomandam brownie cu ciocolata amara'}


3. FLAN-T5 – genereaza raspunsuri(LLM finetunned)

In [14]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-base")

question3_1 = "What would you recommend if I'm craving a cake with salty caramel?"
question3_2 = "Ce imi recomanzi cu caramel?"

print(qa(question3_1))
print(qa(question3_2))

Device set to use cpu


[{'generated_text': "I'd suggest a caramel cake."}]
[{'generated_text': 'io recomanzi cu caramelo?'}]


4. BART – generare si q&a

In [15]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

text = "Transformers sunt folosiți în diverse aplicații NLP, de la clasificare, la generare de text, traducere, și QA."
print(summarizer(text))


config.json: 0.00B [00:00, ?B/s]

D:\facultate\Semestrul 4\AI\TransformereTest\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tapuc\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back t

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 142, but your input_length is only 47. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=23)


[{'summary_text': 'Transformers sunt folosiți în diverse aplicații NLP, de la clasificare, la generare de text, traducere, și QA. Transformers sunt utilizate in diverse formașe,   \xa0de la text-uri \xa0toplicaţi \xa0în NLP.'}]


### Testam pe Dataset

In [24]:
import pandas as pd

df = pd.read_csv("SQuAD-v1.1.csv")
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

for idx, row in df.head(100).iterrows():
    context = row['context']
    question = row['question']
    expected_answer = row['answer']
    
    result = qa(question=question, context=context)
    
    print(f"\n Intrebare: {question}")
    print(f"Context (scurt): {context[:100]}...")
    print(f"Predictie model: {result['answer']}")
    print(f"Raspuns asteptat: {expected_answer}")
    print("-" * 60)


Device set to use cpu



 Intrebare: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: Saint Bernadette Soubirous
Raspuns asteptat: Saint Bernadette Soubirous
------------------------------------------------------------

 Intrebare: What is in front of the Notre Dame Main Building?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: a copper statue of Christ
Raspuns asteptat: a copper statue of Christ
------------------------------------------------------------

 Intrebare: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: the Main Building
Raspuns asteptat: the Main Building
------------------------------

In [19]:
correct = 0
total = 100
for idx, row in df.head(total).iterrows():
    result = qa(question=row['question'], context=row['context'])
    if row['answer'].lower() in result['answer'].lower():
        correct += 1

print(f"\n Acuratete estimata: {correct}/{total} ({100*correct/total:.1f}%)")



 Acuratete estimata: 87/100 (87.0%)


In [20]:
qa = pipeline("question-answering", model="deepset/roberta-base-squad2")
correct = 0
total = 100
for idx, row in df.head(100).iterrows():
    context = row['context']
    question = row['question']
    expected_answer = row['answer']
    
    result = qa(question=question, context=context)
    
    print(f"\n Intrebare: {question}")
    print(f"Context (scurt): {context[:100]}...")
    print(f"Predictie model: {result['answer']}")
    print(f"Raspuns asteptat: {expected_answer}")
    print("-" * 60)
    if row['answer'].lower() in result['answer'].lower():
        correct += 1

print(f"\n Acuratete estimata: {correct}/{total} ({100*correct/total:.1f}%)")


Device set to use cpu



 Intrebare: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: Saint Bernadette Soubirous
Raspuns asteptat: Saint Bernadette Soubirous
------------------------------------------------------------

 Intrebare: What is in front of the Notre Dame Main Building?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: a copper statue of Christ
Raspuns asteptat: a copper statue of Christ
------------------------------------------------------------

 Intrebare: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Context (scurt): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden...
Predictie model: Main Building
Raspuns asteptat: the Main Building
----------------------------------

In [29]:
qa = pipeline("text2text-generation", model="google/flan-t5-base")
correct = 0
total = 100
for idx, row in df.head(100).iterrows():
    question = row['question']
    expected_answer = row['answer']
    
    result = qa(question)
    
    print(f"\n Intrebare: {question}")
    print(f"Predictie model: {result[0]['generated_text'] }")
    print(f"Raspuns asteptat: {expected_answer}")
    print("-" * 60)
    if row['answer'].lower() in result[0]['generated_text'].lower():
        correct += 1

print(f"\n Acuratete estimata: {correct}/{total} ({100*correct/total:.1f}%)")


Device set to use cpu



 Intrebare: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Predictie model: charles de gaulle
Raspuns asteptat: Saint Bernadette Soubirous
------------------------------------------------------------

 Intrebare: What is in front of the Notre Dame Main Building?
Predictie model: Notre Dame Cathedral
Raspuns asteptat: a copper statue of Christ
------------------------------------------------------------

 Intrebare: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Predictie model: st michael s cathedral
Raspuns asteptat: the Main Building
------------------------------------------------------------

 Intrebare: What is the Grotto at Notre Dame?
Predictie model: Notre Dame Notre Dame Grotto
Raspuns asteptat: a Marian place of prayer and reflection
------------------------------------------------------------

 Intrebare: What sits on top of the Main Building at Notre Dame?
Predictie model: tower of notre dame
Raspuns asteptat: a golden